# Chapter 3: Feature Engineering

Wrecksys is a sequential recommendation system. Sequential recommendation systems attempt to predict the next item a user will interact with by examining the history of their past interactions. (TODO: CITATION)

Each subsequence in a user's chronological interaction history provides the context for the machine learning model. The next item a given user interacted with serves as a label for training. (TODO: CITATION)

This chapter creates a simple dataset as a minimum viable product suitable for use in a sequential recommendation system.

## Feature Selection
At a minimum, sequential recommendations require a set of users and a chronological list of items they have interacted with. Wrecksys also incorporates user ratings in an attempt to improve recommendation quality.

Our minimum viable product, therefore, will require four types of data: Users, Books, Ratings, and Timestamps.

The data assembled in Chapter 2 provides the basic fields necessary create the necessary features. 

In [1]:
%%capture
import inspect

from IPython.display import display, Code

from wrecksys_ai.io import load_ratings, load_datasets
from wrecksys_ai.io.load import build_timelines, build_contexts, build_examples, build_datasets
from wrecksys_ai.io.pipeline import sample_record

In [2]:
ratings = load_ratings()
ratings.head()

,user_id,rating,timestamp,work_id
0,6,5.0,1359914868,13262
1,6,4.0,1359939204,3252
2,6,5.0,1359939995,4047
3,6,4.0,1359940272,7070
4,6,5.0,1359940294,19180


The rating data is currently stored with each row representing an interaction between an individual user and a single book. The first step in preparing this data for use in the sequential recommendation algorithm is to create a timeline of interactions for each user.

The above dataframe was created in Chapter 2, where we sorted interactions by user and timestamp prior to saving. Since the data is already sorted chronologically, we don't have to worry about sorting when building user timelines.

In [3]:
display(Code(data=inspect.getsource(build_timelines), language='python3'))

def build_timelines(df=None) -> List[UserHistory]:
    timelines = []
    df = load_ratings() if df is None else df

    with tqdm(total=len(df.user_id.unique()),
              desc="Building user timelines",
              file=sys.stdout,
              unit=' users') as timeline_progress:
        for _, group in df.groupby('user_id', observed=True):
            books = group['work_id'].tolist()
            ratings = group['rating'].tolist()
            size = len(books)
            timelines.append(UserHistory(size, books, ratings))
            timeline_progress.update(1)

    return timelines

In [4]:
timelines = build_timelines(ratings)

Building user timelines:   0%|          | 0/145142 [00:00<?, ? users/s]

Each element of the timeline list represents a single user's rating history. These elements are NamedTuples containing the number of recorded ratings (history), a list of books they have rated, and the ratings they gave each book.

In [5]:
sample_user = timelines[0]
del timelines
sample_user

UserHistory(history=106, books=[13262, 3252, 4047, 7070, 19180, 62, 4789, 4738, 13760, 14072, 13453, 14890, 15786, 16153, 17946, 16370, 16910, 17349, 18154, 6729, 14608, 3912, 591, 1760, 3208, 15696, 21404, 13882, 2465, 4402, 15715, 20752, 22286, 21871, 22501, 17985, 19246, 20047, 12884, 18065, 22923, 20858, 13764, 15670, 17515, 12861, 15805, 18642, 2950, 4885, 5430, 20076, 7362, 20498, 20754, 14048, 7347, 15551, 17683, 18524, 579, 1279, 2204, 388, 4985, 20835, 6827, 601, 2727, 1036, 1967, 2313, 22876, 22965, 17119, 4456, 22442, 22680, 7979, 18819, 4153, 11209, 14330, 4978, 9490, 4067, 9101, 2377, 17116, 148, 22350, 3854, 6441, 7509, 8741, 11968, 22064, 11380, 6383, 3672, 14317, 11826, 18407, 1682, 2172, 4458], ratings=[5.0, 4.0, 5.0, 4.0, 5.0, 5.0, 3.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 3.0, 5.0, 5.0, 3.0, 3.0, 4.0, 5.0, 5.0, 5.0, 3.0, 5.0, 5.0, 4.0, 4.0, 3.0, 3.0, 4.0, 4.0, 4.0, 4.0, 3.0, 3.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 4.0, 5.0, 5.0, 5.0, 3.0, 4.0, 5.0, 

Each user's history is then subdivided into contexts, which are used as training data for our model. Wrecksys currently imposes minimum and maximum length restricts for these contexts. 

A minimum length of 3 allows the algorithm to generate predictions for new users with as few as 3 ratings. It still suffers from the cold start problem common in recommendation systems, but can generate usable predictions for these new users much more quickly than traditional collaborative filtering based approaches. Source: This is the fourth rewrite of Wrecksys. Previous implementations included: user-item collaborative filtering (with and without matrix factorization), and item-item collaborative filtering. This version produces better recommendations with less overhead than its predecessors.

The maximum length of 10 was selected arbitrarily and has produced passable results.

The label for each context is the next book a particular user interacted with. The model attempts to predict the label for each sequence of books during training. Further details are provided in (TODO: CHAPTER ?: Implementation Details) 

In [12]:
display(Code(data=inspect.getsource(build_contexts), language='python3'))

def build_contexts(user: UserHistory) -> List[UserContext]:
    contexts = []
    max_length = model_config.max_series_length
    min_length = model_config.min_series_length

    for label in range(1, len(user.books)):
        pos = max(0, label - max_length)
        if (label - pos) >= min_length:
            context_ids = user.books[pos:label]
            context_ratings = user.ratings[pos:label]
            label_id = user.books[label]
            contexts.append(UserContext(context_ids, context_ratings, label_id))

    return contexts

In [13]:
sample_contexts = build_contexts(sample_user)[0:5]
for row in sample_contexts:
    print(f"User History: {row.ids}")
    print(f"Next Book: {row.label}")

User History: [13262, 3252, 4047]
Next Book: 7070
User History: [13262, 3252, 4047, 7070]
Next Book: 19180
User History: [13262, 3252, 4047, 7070, 19180]
Next Book: 62
User History: [13262, 3252, 4047, 7070, 19180, 62]
Next Book: 4789
User History: [13262, 3252, 4047, 7070, 19180, 62, 4789]
Next Book: 4738


Generated recommendations are based entirely on the sample dataset. The current implementation does not incorporate new user, rating or book data. The shortcomings of this approach are detailed in later chapters.

A fixed dataset simplifies our input pipeline. The above sections generated contexts from available data, which can be serialized to Tensorflow's native TFRecord format.   

In [14]:
sample_user_record = build_examples(sample_contexts)[0]
sample_user_record

features {
  feature {
    key: "label_id"
    value {
      int64_list {
        value: 7070
      }
    }
  }
  feature {
    key: "context_rating"
    value {
      float_list {
        value: 5
        value: 4
        value: 5
        value: 0
        value: 0
        value: 0
        value: 0
        value: 0
        value: 0
        value: 0
      }
    }
  }
  feature {
    key: "context_id"
    value {
      int64_list {
        value: 13262
        value: 3252
        value: 4047
        value: 0
        value: 0
        value: 0
        value: 0
        value: 0
        value: 0
        value: 0
      }
    }
  }
}

We've just examined the steps involved in creating a single training example for our machine learning model. Time to sit back, relax, and do it 14 million more times.

In [15]:
_ = build_datasets()

Building user timelines:   0%|          | 0/145142 [00:00<?, ? users/s]

Converting to tensors:   0%|          | 0/145142 [00:00<?, ? timelines/s]

Writing TFRecords:   0%|          | 0/14414080 [00:00<?, ? records/s]

In [6]:
# Checking the results
_, test = load_datasets()
sample_record(test)

{'context_id': <tf.Tensor: shape=(10,), dtype=int32, numpy=array([1160,  791,  976,    8,  128, 1415, 2073, 1896, 1572,  499])>,
 'context_rating': <tf.Tensor: shape=(10,), dtype=float32, numpy=array([4., 4., 3., 4., 4., 4., 3., 4., 3., 4.], dtype=float32)>,
 'label_id': <tf.Tensor: shape=(1,), dtype=int32, numpy=array([118])>}